# Day 22: Decision Tree Intuition
**Phase 3: Tree Models & SVM**

---
## What is a Decision Tree?

A **Decision Tree** is a supervised learning algorithm used for both **classification** and **regression**. It mimics human decision-making by breaking down data into smaller subsets based on **feature conditions**, forming a tree-like structure.

### Analogy
Think of a **20 Questions game** — each question (node) splits the possibilities until you reach a confident guess (leaf).

### Why Decision Trees?
- ✅ **Interpretable** — easy to explain to non-technical stakeholders
- ✅ **No feature scaling needed** — tree splits are threshold-based, not distance-based
- ✅ **Handles non-linearity** — can capture complex interactions
- ✅ **Feature importance** — built-in feature selection
- ❌ **Prone to overfitting** — needs pruning
- ❌ **High variance** — small data changes can create very different trees

---
## Tree Terminology

| Term | Description |
|------|-------------|
| **Root Node** | Topmost node; represents the entire dataset |
| **Internal Node** | A node that splits into further nodes |
| **Leaf / Terminal Node** | A node that makes the final prediction |
| **Branch / Subtree** | A section of the tree from one node downward |
| **Splitting** | Dividing a node into two or more child nodes |
| **Pruning** | Removing branches to prevent overfitting |
| **Decision Boundary** | The threshold at which a node splits |

```
                    [Root Node]                       
                   /           \                      
          [Internal Node]    [Leaf Node]              
          /            \                              
    [Leaf Node]   [Leaf Node]                         
```

---
## How a Decision Tree is Built (CART Algorithm)

The **CART (Classification and Regression Trees)** algorithm builds binary trees using a **greedy, top-down, recursive** approach:

1. Start with all data at the **root node**
2. For each feature, try every possible split value
3. Choose the **best split** that produces the **purest** child nodes
4. Split the data into two child nodes
5. Repeat steps 2-4 for each child node
6. Stop when a **stopping criterion** is met (max_depth, min_samples_leaf, etc.)

### How does the algorithm choose "best split"?
It uses a **purity/impurity metric** to measure split quality:

---
## Splitting Criteria for Classification

### 1. Gini Impurity

Gini Impurity measures how often a randomly chosen element would be **misclassified** if randomly labeled according to the class distribution.

**Formula:**
$$\text{Gini}(t) = 1 - \sum_{i=1}^{c} p_i^2$$

where $p_i$ is the proportion of samples belonging to class $i$ in node $t$.

| Scenario | Class Proportions | Gini |
|----------|-------------------|------|
| **Pure node** | [1, 0] | 0.0 |
| **Equal split** | [0.5, 0.5] | 0.5 |
| **4 classes equal** | [0.25, 0.25, 0.25, 0.25] | 0.75 |

**Gini ranges from 0 (purest) to ~1 (impurest).**

In [ ]:
# === GINI IMPURITY FROM SCRATCH ===

def gini_impurity(classes):
    """
    Calculate Gini Impurity for a node.
    classes: list of class labels (e.g., [0, 0, 1, 1, 1])
    """
    if len(classes) == 0:
        return 0
    
    total = len(classes)
    counts = {}
    for c in classes:
        counts[c] = counts.get(c, 0) + 1
    
    impurity = 1.0
    for c in counts:
        p = counts[c] / total
        impurity -= p ** 2
    
    return impurity

# Example calculations
pure = [0, 0, 0]  # all same class
equal = [0, 1]    # 50-50 split
mixed = [0, 0, 0, 1, 1, 1, 2, 2]  # 3 classes

print(f"Pure node [0,0,0]: Gini = {gini_impurity(pure):.4f}")
print(f"Equal split [0,1]: Gini = {gini_impurity(equal):.4f}")
print(f"Mixed 3-class:    Gini = {gini_impurity(mixed):.4f}")

# scikit-learn's Gini for verification
from sklearn.metrics import accuracy_score
print("\n→ Gini = 0 when node is perfectly pure")
print("→ Gini increases as node becomes more mixed")

---
### 2. Entropy & Information Gain

**Entropy** measures the **amount of uncertainty/randomness** in a node.

**Formula:**
$$\text{Entropy}(t) = -\sum_{i=1}^{c} p_i \log_2(p_i)$$

**Information Gain (IG)** measures how much entropy is reduced by splitting on a feature:

$$\text{IG}(S, f) = \text{Entropy}(S) - \sum_{v \in \text{Values}(f)} \frac{|S_v|}{|S|} \cdot \text{Entropy}(S_v)$$

| Scenario | Class Proportions | Entropy |
|----------|-------------------|---------|
| **Pure node** | [1, 0] | 0.0 |
| **Equal split** | [0.5, 0.5] | 1.0 |
| **4 classes equal** | [0.25, 0.25, 0.25, 0.25] | 2.0 |

**Entropy ranges from 0 (purest) to log₂(c) (impurest).**

### Gini vs Entropy — Which to use?
- **Gini** is faster to compute (no log), tends to isolate the majority class faster
- **Entropy** produces more balanced trees
- In practice, **the difference is minimal** — try both

In [ ]:
# === ENTROPY & INFORMATION GAIN FROM SCRATCH ===
import math

def entropy(classes):
    """Calculate Entropy for a node."""
    if len(classes) == 0:
        return 0
    total = len(classes)
    counts = {}
    for c in classes:
        counts[c] = counts.get(c, 0) + 1
    
    ent = 0.0
    for c in counts:
        p = counts[c] / total
        if p > 0:
            ent -= p * math.log2(p)
    return ent

def information_gain(parent, left_child, right_child):
    """
    Calculate Information Gain after a binary split.
    IG = Entropy(parent) - weighted_avg(Entropy(child1), Entropy(child2))
    """
    parent_entropy = entropy(parent)
    total = len(left_child) + len(right_child)
    
    weighted = (len(left_child) / total) * entropy(left_child) + \
               (len(right_child) / total) * entropy(right_child)
    
    return parent_entropy - weighted

# Demo: Split on "Age >= 30?"
parent = ["No", "No", "Yes", "Yes", "Yes", "No", "Yes", "No"]
left = ["No", "No", "No", "No"]    # Age < 30 (4 samples, all No)
right = ["Yes", "Yes", "Yes", "No"]  # Age >= 30 (4 samples, 3 Yes, 1 No)

print(f"Parent Entropy: {entropy(parent):.4f}")
print(f"Left Child Entropy:  {entropy(left):.4f}")
print(f"Right Child Entropy: {entropy(right):.4f}")
ig = information_gain(parent, left, right)
print(f"\nInformation Gain from split: {ig:.4f}")
print(f"→ Higher IG = better split")

---
## Splitting for Regression (MSE)

For **regression trees**, instead of Gini/Entropy, we use **Mean Squared Error (MSE)** or **Mean Absolute Error (MAE)** as the splitting criterion.

$$\text{MSE}(t) = \frac{1}{N_t} \sum_{i \in t} (y_i - \bar{y}_t)^2$$

where $\bar{y}_t$ is the mean target value in node $t$.

The best split minimizes the weighted MSE of child nodes:

In [ ]:
# === REGRESSION SPLITTING WITH MSE ===
import numpy as np

def mse(values):
    """Calculate MSE for a set of target values."""
    if len(values) == 0:
        return 0
    mean = np.mean(values)
    return np.mean((values - mean) ** 2)

# Example: predicting house price based on a split
parent_prices = np.array([100, 120, 150, 180, 200, 220, 250, 300])
left_prices = np.array([100, 120, 150, 180])    # e.g., bedrooms < 3
right_prices = np.array([200, 220, 250, 300])   # bedrooms >= 3

parent_mse = mse(parent_prices)
weighted_child_mse = (len(left_prices)/len(parent_prices))*mse(left_prices) + \
                     (len(right_prices)/len(parent_prices))*mse(right_prices)

print(f"Parent MSE: {parent_mse:.2f}")
print(f"Left child MSE:  {mse(left_prices):.2f}")
print(f"Right child MSE: {mse(right_prices):.2f}")
print(f"Weighted child MSE: {weighted_child_mse:.2f}")
print(f"MSE reduction: {parent_mse - weighted_child_mse:.2f}")

---
## Decision Tree with sklearn — Quick Demo

In [ ]:
# === DECISION TREE CLASSIFICATION DEMO ===

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Load data
iris = load_iris()
X, y = iris.data, iris.target
feature_names = iris.feature_names
class_names = list(iris.target_names)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Train Decision Tree (with constraints to prevent overfitting)
dt = DecisionTreeClassifier(max_depth=3, criterion='gini', random_state=42)
dt.fit(X_train, y_train)

# Evaluate
train_acc = dt.score(X_train, y_train)
test_acc = dt.score(X_test, y_test)
print(f"Training Accuracy:   {train_acc:.4f}")
print(f"Test Accuracy:       {test_acc:.4f}")

y_pred = dt.predict(X_test)
print(f"\nClassification Report:\n{classification_report(y_test, y_pred, target_names=class_names)}")

In [ ]:
# === VISUALIZE THE DECISION TREE ===

plt.figure(figsize=(20, 10))
plot_tree(dt, filled=True, feature_names=feature_names, class_names=class_names, rounded=True, fontsize=12)
plt.title("Decision Tree — Iris Dataset (max_depth=3)", fontsize=16)
plt.show()

# Interpretation: trace the path for a sample
print("\n--- How to read the tree ---")
print("- Each node shows: splitting condition, gini, samples, value, class")
print("- Blue = class 0 (setosa), Orange = class 1 (versicolor), Green = class 2 (virginica)")
print("- Darker color = higher purity at that node")

In [ ]:
# === FEATURE IMPORTANCE ===

importances = pd.Series(dt.feature_importances_, index=feature_names).sort_values(ascending=False)

plt.figure(figsize=(8, 4))
importances.plot(kind='bar', color='skyblue', edgecolor='black')
plt.title("Feature Importance — Decision Tree")
plt.ylabel("Importance")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("Feature Importances:")
for f, imp in importances.items():
    print(f"  {f}: {imp:.4f}")
print("\nNote: Feature importance is computed as the total reduction in Gini impurity")
print("weighted by the proportion of samples reaching that node.")

---
## Key Hyperparameters for Decision Trees

| Parameter | Purpose | Default | Effect |
|-----------|---------|---------|--------|
| `max_depth` | Maximum tree depth | None | Lower = simpler, higher = more complex |
| `min_samples_split` | Min samples to split a node | 2 | Higher = prevents overfitting |
| `min_samples_leaf` | Min samples in a leaf | 1 | Higher = smoother boundaries |
| `max_features` | Features to consider per split | None | Lower = more randomness |
| `criterion` | Split quality metric | 'gini' | 'gini' or 'entropy' |
| `ccp_alpha` | Cost-complexity pruning | 0.0 | Higher = more pruning |
| `random_state` | Reproducibility | None | Set for deterministic results |

### Overfitting vs Underfitting
```
Underfitting ← → Overfitting
    max_depth=1      max_depth=None
    Large bias        Low bias
    Low variance      High variance
```

In [ ]:
# === EFFECT OF max_depth ON PERFORMANCE ===

depths = range(1, 11)
train_scores = []
test_scores = []

for depth in depths:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_train, y_train)
    train_scores.append(dt.score(X_train, y_train))
    test_scores.append(dt.score(X_test, y_test))

plt.figure(figsize=(10, 5))
plt.plot(depths, train_scores, 'bo-', label='Training Accuracy')
plt.plot(depths, test_scores, 'rs-', label='Test Accuracy')
plt.xlabel('max_depth')
plt.ylabel('Accuracy')
plt.title('Effect of max_depth on Overfitting')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(depths)
plt.show()

print("Observation:")
print("- As max_depth increases, training accuracy approaches 100%")
print("- Test accuracy peaks at a depth and then may drop — that's overfitting")
print("- The optimal depth balances bias and variance")

---
## Decision Tree for Regression

In [ ]:
# === DECISION TREE REGRESSION DEMO ===

from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Generate synthetic regression data
np.random.seed(42)
X_reg = np.sort(5 * np.random.rand(80, 1), axis=0)
y_reg = np.sin(X_reg).ravel() + np.random.normal(0, 0.1, X_reg.shape[0])

X_reg_train, X_reg_test = X_reg[:60], X_reg[60:]
y_reg_train, y_reg_test = y_reg[:60], y_reg[60:]

# Train Decision Tree Regressor with different depths
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, depth in enumerate([1, 3, 10]):
    dtr = DecisionTreeRegressor(max_depth=depth, random_state=42)
    dtr.fit(X_reg_train, y_reg_train)
    
    X_plot = np.linspace(0, 5, 100).reshape(-1, 1)
    y_pred = dtr.predict(X_plot)
    
    axes[idx].scatter(X_reg_train, y_reg_train, c='blue', label='Train', alpha=0.6)
    axes[idx].scatter(X_reg_test, y_reg_test, c='red', label='Test', alpha=0.6)
    axes[idx].plot(X_plot, y_pred, 'g-', linewidth=2, label='Prediction')
    axes[idx].set_title(f'max_depth={depth}')
    axes[idx].set_xlabel('X')
    axes[idx].set_ylabel('y')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)
    
    train_mse = mean_squared_error(y_reg_train, dtr.predict(X_reg_train))
    test_mse = mean_squared_error(y_reg_test, dtr.predict(X_reg_test))
    print(f"max_depth={depth}: Train MSE={train_mse:.4f}, Test MSE={test_mse:.4f}")

plt.tight_layout()
plt.show()

print("\n→ depth=1: Underfitting (high bias, both errors high)")
print("→ depth=3: Good balance (low errors on both)")
print("→ depth=10: Overfitting (train ~0, test high)")

---
## Limitations of Decision Trees

| Limitation | Why it happens | Solution |
|------------|---------------|----------|
| **High variance** | Small data changes → completely different tree | Ensemble methods (Random Forest) |
| **Overfitting** | Tree can grow until every leaf is pure | Pruning, max_depth, min_samples_leaf |
| **Bias towards features with more levels** | More splits possible → higher chance of being selected | Feature selection, domain knowledge |
| **Instability** | Orthogonal decision boundaries (axis-aligned splits) | Ensemble methods, feature scaling |
| **Poor extrapolation** | Cannot predict beyond training range of target | Use other models for extrapolation |

### Strengths
- ✅ **White-box model** — completely interpretable
- ✅ **No data preprocessing** — handles mixed data types
- ✅ **Non-parametric** — no assumption about data distribution
- ✅ **Handles non-linearity and interactions** automatically

---
## Practice Exercises

### Exercise 1: Manual Gini Calculation
```
Given node with: [Red, Red, Blue, Blue, Blue, Green]
Calculate Gini Impurity manually.
```

### Exercise 2: Manual Information Gain
```
Parent: [Yes, No, Yes, Yes, No, No] (6 samples)
Split on Feature X:
  Left child (X < 5):  [Yes, Yes, No]  (3 samples)
  Right child (X >= 5): [Yes, No, No]   (3 samples)
Calculate Information Gain.
```

### Exercise 3: Train a DT on a Real Dataset
```
1. Load the Titanic or Breast Cancer dataset
2. Train a DecisionTreeClassifier
3. Find the best max_depth using grid search
4. Visualize the tree
5. Report train vs test accuracy
6. List top 3 features by importance
```

### Exercise 4: Decision Surface Visualization
```
For the Iris dataset using only 2 features (petal length, petal width):
1. Train a DT with max_depth=2
2. Plot the decision boundary
3. Plot a DT with max_depth=5
4. Compare how boundaries change with depth
```

### Exercise 5: Tree Comparison
```
Compare Decision Tree performance (accuracy, training time, tree size)
using:
- criterion = 'gini'
- criterion = 'entropy'
- With and without max_depth limitation
- With min_samples_leaf = 1 vs 5 vs 10
```

---
## Summary — What You Learned Today

✅ What a Decision Tree is and why it's useful
✅ Tree terminology: root, internal nodes, leaves, branches
✅ CART algorithm — greedy, top-down, recursive binary splitting
✅ Gini Impurity — formula, manual calculation, interpretation (0 = pure)
✅ Entropy & Information Gain — formula, manual calculation
✅ MSE splitting for regression trees
✅ sklearn DecisionTreeClassifier and DecisionTreeRegressor
✅ Tree visualization with plot_tree
✅ Feature importance extraction
✅ Hyperparameters: max_depth, min_samples_split, min_samples_leaf
✅ Overfitting vs underfitting in trees
✅ Decision surface for different depths
✅ Limitations and when to use Decision Trees